# 05 构建三种数据划分

本 notebook 从 `outputs/sampled_data/<数据集>/1_<倍数>/` 读取 04 输出的五个随机采样版本，并将三种划分独立保存到 `outputs/split_data/<数据集>/1_<倍数>/<划分策略>/seed_<种子>/`。random 按标签分层随机划分样本；cold protein 保证蛋白质不跨集合；cold probe 保证探针不跨集合。

In [29]:
# Cell 1：配置
import os
import random
import pandas as pd

NEGATIVE_RATIO = 10
SAMPLE_SEEDS = [42, 123, 2024, 3407, 8888]
SPLIT_SEED = 42
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
SAMPLED_ROOT = os.path.join("outputs", "sampled_data")
SPLIT_ROOT = os.path.join("outputs", "split_data")
RATIO_DIRNAME = f"1_{NEGATIVE_RATIO}"
DATASET_NAMES = ["Human_Lysate_100uM", "Human_Lysate_1mM",
                 "SA59_Lysate_100uM", "SA59_Lysate_1mM", "SA59_Insitu_100uM"]
COLUMNS = ["Probe", "Uniprot_ID", "Site", "Postion", "Label"]
os.makedirs(SPLIT_ROOT, exist_ok=True)

In [30]:
# Cell 2：划分辅助函数
def partition_sizes(n):
    if n < 3:
        return n, 0, 0
    n_val = max(1, round(n * SPLIT_RATIOS["val"]))
    n_test = max(1, round(n * SPLIT_RATIOS["test"]))
    n_train = n - n_val - n_test
    if n_train < 1:
        n_train, n_val, n_test = n - 2, 1, 1
    return n_train, n_val, n_test

def assign_items(items, seed):
    values = list(items)
    random.Random(seed).shuffle(values)
    n_train, n_val, _ = partition_sizes(len(values))
    return {
        "train": set(values[:n_train]),
        "val": set(values[n_train:n_train + n_val]),
        "test": set(values[n_train + n_val:]),
    }

def random_stratified_split(frame, seed):
    index_sets = {name: [] for name in SPLIT_RATIOS}
    for label, group in frame.groupby("Label", sort=True):
        assignment = assign_items(group.index.tolist(), seed + int(label))
        for split_name, indices in assignment.items():
            index_sets[split_name].extend(indices)
    return {name: frame.loc[indices, COLUMNS].sample(frac=1, random_state=seed).reset_index(drop=True) for name, indices in index_sets.items()}

def grouped_split(frame, column, seed):
    assignment = assign_items(frame[column].drop_duplicates().tolist(), seed)
    return {name: frame[frame[column].isin(values)][COLUMNS].reset_index(drop=True) for name, values in assignment.items()}

In [31]:
# Cell 3：生成并保存三种划分
all_splits, summary_rows = {}, []
for sampling_seed in SAMPLE_SEEDS:
    for dataset_index, dataset_name in enumerate(DATASET_NAMES):
        sampled_dataset_root = os.path.join(SAMPLED_ROOT, dataset_name, RATIO_DIRNAME)
        split_dataset_root = os.path.join(SPLIT_ROOT, dataset_name, RATIO_DIRNAME)
        input_path = os.path.join(sampled_dataset_root, f"seed_{sampling_seed}.csv")
        frame = pd.read_csv(input_path)
        assert list(frame.columns) == COLUMNS
        strategy_tables = {
            "random": random_stratified_split(frame, SPLIT_SEED + 1000 * dataset_index),
            "cold_protein": grouped_split(frame, "Uniprot_ID", SPLIT_SEED + 1000 * dataset_index),
        }
        if frame["Probe"].nunique() >= 3:
            strategy_tables["cold_probe"] = grouped_split(frame, "Probe", SPLIT_SEED + 1000 * dataset_index)
        else:
            summary_rows.append({"Sampling_seed": sampling_seed, "Dataset": dataset_name, "Strategy": "cold_probe", "Split": "not_applicable", "Rows": 0, "Positive_sites": 0, "Unlabeled_sites": 0, "Proteins": frame["Uniprot_ID"].nunique(), "Probes": frame["Probe"].nunique(), "Status": "requires_at_least_3_probes"})

        all_splits[(sampling_seed, dataset_name)] = strategy_tables
        for strategy, tables in strategy_tables.items():
            output_dir = os.path.join(split_dataset_root, strategy, f"seed_{sampling_seed}")
            os.makedirs(output_dir, exist_ok=True)
            for split_name, table in tables.items():
                table.to_csv(os.path.join(output_dir, f"{split_name}.csv"), index=False, encoding="utf-8-sig")
                summary_rows.append({"Sampling_seed": sampling_seed, "Dataset": dataset_name, "Strategy": strategy, "Split": split_name, "Rows": len(table), "Positive_sites": int(table["Label"].sum()), "Unlabeled_sites": int((table["Label"] == 0).sum()), "Proteins": table["Uniprot_ID"].nunique(), "Probes": table["Probe"].nunique(), "Status": "ok"})

In [32]:
# Cell 4：验证完整性与 cold-split 无泄漏
sample_key = ["Probe", "Uniprot_ID", "Site", "Postion"]
for (sampling_seed, dataset_name), strategy_tables in all_splits.items():
    source = pd.read_csv(os.path.join(SAMPLED_ROOT, dataset_name, RATIO_DIRNAME, f"seed_{sampling_seed}.csv"))
    source_keys = set(map(tuple, source[sample_key].itertuples(index=False, name=None)))
    for strategy, tables in strategy_tables.items():
        split_keys = {name: set(map(tuple, table[sample_key].itertuples(index=False, name=None))) for name, table in tables.items()}
        assert not split_keys["train"] & split_keys["val"]
        assert not split_keys["train"] & split_keys["test"]
        assert not split_keys["val"] & split_keys["test"]
        assert set().union(*split_keys.values()) == source_keys
        if strategy == "cold_protein":
            groups = {name: set(table["Uniprot_ID"]) for name, table in tables.items()}
            assert not groups["train"] & groups["val"]
            assert not groups["train"] & groups["test"]
            assert not groups["val"] & groups["test"]
        if strategy == "cold_probe":
            groups = {name: set(table["Probe"]) for name, table in tables.items()}
            assert not groups["train"] & groups["val"]
            assert not groups["train"] & groups["test"]
            assert not groups["val"] & groups["test"]

split_summary = pd.DataFrame(summary_rows)
split_summary.to_csv(os.path.join(SPLIT_ROOT, f"split_summary_{RATIO_DIRNAME}.csv"), index=False, encoding="utf-8-sig")
split_summary

,Sampling_seed,Dataset,Strategy,Split,Rows,Positive_sites,Unlabeled_sites,Proteins,Probes,Status
0,42,Human_Lysate_100uM,random,train,75725,11513,64212,3054,13,ok
1,42,Human_Lysate_100uM,random,val,16227,2467,13760,2629,13,ok
2,42,Human_Lysate_100uM,random,test,16227,2467,13760,2639,13,ok
3,42,Human_Lysate_100uM,cold_protein,train,74446,11252,63194,2153,13,ok
4,42,Human_Lysate_100uM,cold_protein,val,15388,2491,12897,462,13,ok
...,...,...,...,...,...,...,...,...,...,...
210,8888,SA59_Insitu_100uM,cold_protein,val,5808,874,4934,134,13,ok
211,8888,SA59_Insitu_100uM,cold_protein,test,4874,757,4117,134,13,ok
212,8888,SA59_Insitu_100uM,cold_probe,train,32571,5303,27268,826,10,ok
213,8888,SA59_Insitu_100uM,cold_probe,val,1165,171,994,111,2,ok
